# FreshOrRotten - Baseline CNN Google Colab

In [ ]:
from pathlib import Path
import os
import subprocess

repo_url = "https://github.com/Anthony2210/FreshOrRotten.git"
repo_dir = Path("/content/FreshOrRotten")

if repo_dir.exists():
    os.chdir(repo_dir)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)
    os.chdir(repo_dir)

print("Dossier courant :", Path.cwd())

In [ ]:
import subprocess

subprocess.run(["python", "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

# Dossier source dans Google Drive.
drive_dataset_dir = Path("/content/drive/MyDrive/data/raw")

# Dossier local rapide dans la session Colab.
local_dataset_dir = Path("/content/local_raw")

if not drive_dataset_dir.exists():
    raise FileNotFoundError(f"Dataset introuvable dans Drive : {drive_dataset_dir}")

if local_dataset_dir.exists():
    shutil.rmtree(local_dataset_dir)

local_dataset_dir.mkdir(parents=True, exist_ok=True)

# On copie les dossiers de catégories dans le disque local Colab.
for item in drive_dataset_dir.iterdir():
    destination = local_dataset_dir / item.name
    if item.is_dir():
        shutil.copytree(item, destination)
    else:
        shutil.copy2(item, destination)

category_names = sorted(path.name for path in local_dataset_dir.iterdir() if path.is_dir())
print("Dossier local :", local_dataset_dir)
print("Nombre de dossiers de catégories :", len(category_names))
print("Exemples :", category_names[:10])

In [ ]:
from pathlib import Path
import yaml

config_path = Path("config.yaml")
config = yaml.safe_load(config_path.read_text(encoding="utf-8"))

config["paths"]["raw_data_dir"] = str(local_dataset_dir)
config["training"]["batch_size"] = 64
config["training"]["early_stopping_patience"] = 3

config.setdefault("generalization", {})
config["generalization"]["unseen_model_filename"] = "baseline_model_unseen.keras"
config["generalization"]["unseen_history_filename"] = "unseen_training_history.csv"
config["generalization"]["unseen_split_filename"] = "unseen_category_split.csv"
config["generalization"]["unseen_metrics_filename"] = "unseen_category_evaluation_metrics.csv"
config["generalization"]["unseen_confusion_matrix_filename"] = "unseen_category_confusion_matrix.csv"
config["generalization"]["unseen_category_metrics_filename"] = "unseen_category_metrics_by_product_type.csv"
config["generalization"]["comparison_filename"] = "evaluation_comparison.csv"

config_path.write_text(
    yaml.safe_dump(config, sort_keys=False, allow_unicode=True),
    encoding="utf-8",
)

print(config_path.read_text(encoding="utf-8"))

In [ ]:
import subprocess

subprocess.run(["python", "src/train.py", "--help"], check=True)
subprocess.run(["python", "src/evaluate.py", "--help"], check=True)

In [ ]:
from pathlib import Path
import pandas as pd

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
image_paths = [path for path in local_dataset_dir.rglob("*") if path.suffix.lower() in image_extensions]

folder_counts = []
for folder in sorted(path for path in local_dataset_dir.iterdir() if path.is_dir()):
    image_count = sum(1 for path in folder.rglob("*") if path.suffix.lower() in image_extensions)
    folder_counts.append({"folder": folder.name, "image_count": image_count})

folder_counts_df = pd.DataFrame(folder_counts)
print("Nombre total d'images :", len(image_paths))
display(folder_counts_df.head(15))

In [ ]:
import subprocess

run_standard_training = True

if run_standard_training:
    subprocess.run(["python", "src/train.py"], check=True)
else:
    print("Entraînement standard ignoré.")

In [ ]:
import subprocess

subprocess.run(["python", "src/evaluate.py"], check=True)

In [ ]:
unseen_categories = ["apple", "banana", "tomato"]

print("Catégories non vues :", unseen_categories)

In [ ]:
import subprocess

unseen_command = [
    "python",
    "src/train.py",
    "--split",
    "unseen",
    "--unseen-categories",
    *unseen_categories,
]

subprocess.run(unseen_command, check=True)

In [ ]:
import subprocess

unseen_evaluate_command = [
    "python",
    "src/evaluate.py",
    "--split",
    "unseen",
    "--unseen-categories",
    *unseen_categories,
]

subprocess.run(unseen_evaluate_command, check=True)

In [ ]:
from pathlib import Path
import pandas as pd

reports_dir = Path("reports")

files_to_display = {
    "standard_metrics": reports_dir / "evaluation_metrics.csv",
    "standard_confusion_matrix": reports_dir / "confusion_matrix.csv",
    "unseen_metrics": reports_dir / "unseen_category_evaluation_metrics.csv",
    "unseen_confusion_matrix": reports_dir / "unseen_category_confusion_matrix.csv",
    "unseen_by_product_type": reports_dir / "unseen_category_metrics_by_product_type.csv",
    "comparison": reports_dir / "evaluation_comparison.csv",
}

for title, path in files_to_display.items():
    print("\n===", title, "===")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Fichier absent :", path)


In [ ]:
from pathlib import Path
import shutil

output_dir = Path("/content/drive/MyDrive/FreshOrRotten_outputs")
reports_output_dir = output_dir / "reports"
models_output_dir = output_dir / "models"

reports_output_dir.mkdir(parents=True, exist_ok=True)
models_output_dir.mkdir(parents=True, exist_ok=True)

for path in Path("reports").glob("*"):
    destination = reports_output_dir / path.name
    if path.is_dir():
        if destination.exists():
            shutil.rmtree(destination)
        shutil.copytree(path, destination)
    else:
        shutil.copy2(path, destination)

for path in Path("models").glob("*"):
    if path.is_file():
        shutil.copy2(path, models_output_dir / path.name)

print("Résultats sauvegardés dans :", output_dir)